# 05 — Pipeline B: DSP Preprocessing → AI Model
// Experiments with DSP-processed audio

**Objective**: Train and evaluate all models on DSP-preprocessed audio.

Pipeline B applies: Bandpass filter → Pre-emphasis → Normalization → Feature extraction

Models:
- SVM (handcrafted features from processed audio)
- Random Forest (handcrafted features from processed audio)
- CNN-2D (mel spectrogram from processed audio)

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, ".")


import numpy as np
import pickle
import os
from tqdm import tqdm

import config
from src.data_loader import load_metadata, load_fold_data
from src.dsp_pipeline import pipeline_b_process
from src.feature_extraction import extract_all_features, extract_mel_spectrogram
from src.models.classical_ml import train_svm, train_random_forest
from src.models.deep_learning import CNN2D, train_cnn, get_device, evaluate, AudioDataset
from src.evaluation import compute_metrics, aggregate_fold_results
from src.visualization import plot_training_curves

from torch.utils.data import DataLoader
import torch
import matplotlib.pyplot as plt

plt.style.use('dark_background')
%matplotlib inline

## 1. Setup

In [ ]:
metadata = load_metadata()
np.random.seed(config.RANDOM_SEED)
torch.manual_seed(config.RANDOM_SEED)

svm_fold_metrics = []
rf_fold_metrics = []
cnn_fold_metrics = []

## 2. 10-Fold Cross-Validation — Classical ML (with DSP)

In [ ]:
for test_fold in range(1, config.N_FOLDS + 1):
    print(f"\n=== Fold {test_fold} ===")
    
    train_folds = [f for f in range(1, 11) if f != test_fold]
    X_train_raw, y_train = load_fold_data(metadata, train_folds, show_progress=False)
    X_test_raw, y_test = load_fold_data(metadata, [test_fold], show_progress=False)
    
    # Apply Pipeline B
    X_train_dsp = np.array([pipeline_b_process(x) for x in X_train_raw])
    X_test_dsp = np.array([pipeline_b_process(x) for x in X_test_raw])
    
    # Extract features from processed audio
    feat_train = extract_all_features(X_train_dsp, show_progress=False)
    feat_test = extract_all_features(X_test_dsp, show_progress=False)
    
    # SVM
    svm_model, svm_params, _ = train_svm(feat_train, y_train)
    svm_preds = svm_model.predict(feat_test)
    svm_metrics = compute_metrics(y_test, svm_preds)
    svm_fold_metrics.append(svm_metrics)
    print(f"  SVM Accuracy: {svm_metrics['accuracy']:.4f}")
    
    # Random Forest
    rf_model, rf_params, _, scaler = train_random_forest(feat_train, y_train)
    rf_preds = rf_model.predict(scaler.transform(feat_test))
    rf_metrics = compute_metrics(y_test, rf_preds)
    rf_fold_metrics.append(rf_metrics)
    print(f"  RF  Accuracy: {rf_metrics['accuracy']:.4f}")

## 3. 10-Fold Cross-Validation — CNN-2D (with DSP)

In [ ]:
device = get_device()
print(f"Using device: {device}")

for test_fold in range(1, config.N_FOLDS + 1):
    print(f"\n=== CNN Fold {test_fold} ===")
    
    train_folds = [f for f in range(1, 11) if f != test_fold]
    X_train_raw, y_train = load_fold_data(metadata, train_folds, show_progress=False)
    X_test_raw, y_test = load_fold_data(metadata, [test_fold], show_progress=False)
    
    # Apply Pipeline B
    X_train_dsp = np.array([pipeline_b_process(x) for x in X_train_raw])
    X_test_dsp = np.array([pipeline_b_process(x) for x in X_test_raw])
    
    # Extract mel spectrograms
    X_train_mel = np.array([extract_mel_spectrogram(x) for x in X_train_dsp])
    X_test_mel = np.array([extract_mel_spectrogram(x) for x in X_test_dsp])
    
    # Train CNN
    model = CNN2D()
    model, history = train_cnn(model, X_train_mel, y_train, X_test_mel, y_test)
    
    # Evaluate
    test_dataset = AudioDataset(X_test_mel, y_test)
    test_loader = DataLoader(test_dataset, batch_size=config.CNN_BATCH_SIZE, shuffle=False)
    criterion = torch.nn.CrossEntropyLoss()
    _, _, preds, labels = evaluate(model, test_loader, criterion, device)
    
    cnn_metrics = compute_metrics(labels, preds)
    cnn_fold_metrics.append(cnn_metrics)
    print(f"  CNN Accuracy: {cnn_metrics['accuracy']:.4f}")
    
    if test_fold == 1:
        plot_training_curves(history, title='CNN-2D Training (Pipeline B, Fold 1)',
                           save_name='fig_training_curves_cnn_pipeline_b.png')
        plt.show()

## 4. Aggregate Results

In [ ]:
svm_agg = aggregate_fold_results(svm_fold_metrics)
rf_agg = aggregate_fold_results(rf_fold_metrics)
cnn_agg = aggregate_fold_results(cnn_fold_metrics)

print("=== Pipeline B Results ===")
for name, agg in [('SVM', svm_agg), ('Random Forest', rf_agg), ('CNN-2D', cnn_agg)]:
    acc = agg['accuracy']
    f1 = agg['f1']
    print(f"{name}: Accuracy = {acc['mean']*100:.1f} ± {acc['ci_95']*100:.1f}%, "
          f"F1 = {f1['mean']*100:.1f} ± {f1['ci_95']*100:.1f}%")

# Save results
pipeline_b_results = {
    'svm': svm_fold_metrics,
    'rf': rf_fold_metrics,
    'cnn': cnn_fold_metrics,
    'svm_agg': svm_agg,
    'rf_agg': rf_agg,
    'cnn_agg': cnn_agg,
}
with open(os.path.join(config.RESULTS_DIR, 'pipeline_b_results.pkl'), 'wb') as f:
    pickle.dump(pipeline_b_results, f)
print("\nResults saved.")